# PY200 — Calculating the Median

The **median** is the **middle value** when data is sorted. It divides the dataset into two equal halves.

## Formula
```
If n is odd  → Median = middle element = data[n // 2]
If n is even → Median = average of two middle elements = (data[n//2 - 1] + data[n//2]) / 2
```

## Why Median Matters
- **Robust to outliers** — unlike the mean, a single extreme value does not distort the median.
- Used for income statistics, house prices, response times — anywhere data is skewed.

## Implementations Covered
1. **Manual sort + index** — first principles, O(n log n)
2. **Using `sorted()` built-in** — Pythonic, O(n log n)
3. **Using `statistics.median()`** — standard library
4. **Without sorting: Quickselect algorithm** — O(n) average case

> **Perspective:** The obvious approach to finding a median requires sorting, which is O(n log n). But in theory, you can find the median in O(n) using selection algorithms. Understanding both reveals the difference between *correct* and *optimal*.

---
## Approach 1: Manual Sort (Bubble Sort) + Middle Index — First Principles

To truly understand from scratch, we implement our own sort, then pick the middle element.

**Time Complexity:** O(n²) for bubble sort + O(1) for index lookup = O(n²)  
**Space Complexity:** O(1) — sorts in place

In [ ]:
## Median from first principles — manual Bubble Sort + index

data = [9, 3, 7, 5, 1, 8, 2]

# Step 1: Sort using Bubble Sort (simple but slow)
# Bubble Sort repeatedly swaps adjacent elements if they are in the wrong order.
sorted_data = data.copy()    # Don't modify the original — good habit!

n = len(sorted_data)
for i in range(n):
    for j in range(0, n - i - 1):        # Last i elements are already sorted
        if sorted_data[j] > sorted_data[j + 1]:
            # Swap using Python's tuple unpacking
            sorted_data[j], sorted_data[j + 1] = sorted_data[j + 1], sorted_data[j]

print(f"Original : {data}")
print(f"Sorted   : {sorted_data}")

# Step 2: Pick the middle element
mid = n // 2
if n % 2 == 1:
    # Odd count — single middle element
    median = sorted_data[mid]
else:
    # Even count — average of two middle elements
    median = (sorted_data[mid - 1] + sorted_data[mid]) / 2

print(f"n = {n}, mid index = {mid}")
print(f"Median   : {median}")

# Learning Point: Bubble Sort is O(n²) — fine for learning, terrible for
# production. For a list of 100,000 elements, it makes ~5 billion comparisons.
# Python's built-in sorted() uses Timsort: O(n log n) and much faster.

---
## Approach 2: Using `sorted()` Built-in — The Practical Way

Python's `sorted()` uses **Timsort** — a hybrid merge-sort / insertion-sort that is highly optimized.

**Time Complexity:** O(n log n) for sorting + O(1) for indexing  
**Space Complexity:** O(n) — `sorted()` returns a new list

In [ ]:
## Median using sorted() — clean and reliable

def median_sorted(data: list) -> float:
    """Calculate median using Python's built-in sorted()."""
    if not data:
        raise ValueError("Cannot compute median of empty list")
    
    s = sorted(data)           # O(n log n) — Timsort
    n = len(s)
    mid = n // 2
    
    if n % 2 == 1:
        return float(s[mid])           # Odd: single middle value
    else:
        return (s[mid - 1] + s[mid]) / 2   # Even: average of two middles


# Test with odd and even length lists
odd_data = [9, 3, 7, 5, 1, 8, 2]
even_data = [9, 3, 7, 5, 1, 8]

print(f"Odd  {odd_data}  → Median: {median_sorted(odd_data)}")
print(f"Even {even_data} → Median: {median_sorted(even_data)}")

# Best Practice: Always return float for the median, even if the data is
# all integers. The average of two middle elements can be a float (e.g., 5.5).
# This makes the return type consistent and predictable.

In [ ]:
## Demonstrating odd vs even behavior clearly

# Odd count: [1, 2, 3, 5, 7, 8, 9] → middle = index 3 → value 5
odd = sorted([9, 3, 7, 5, 1, 8, 2])
print(f"Sorted: {odd}")
print(f"Length: {len(odd)} (odd) → Middle index: {len(odd)//2} → Median: {odd[len(odd)//2]}")

print("---")

# Even count: [1, 3, 5, 7, 8, 9] → middle = index 2 & 3 → (5+7)/2 = 6.0
even = sorted([9, 3, 7, 5, 1, 8])
print(f"Sorted: {even}")
mid = len(even) // 2
print(f"Length: {len(even)} (even) → Middle indices: {mid-1} & {mid} → "
      f"Values: {even[mid-1]} & {even[mid]} → Median: {(even[mid-1] + even[mid]) / 2}")

---
## Approach 3: Using `statistics.median()` — Standard Library

The `statistics` module handles all edge cases and type conversions.

**Time Complexity:** O(n log n) — internally sorts the data.

In [ ]:
## Median using the statistics module

import statistics

odd_data = [9, 3, 7, 5, 1, 8, 2]
even_data = [9, 3, 7, 5, 1, 8]

print(f"statistics.median (odd) : {statistics.median(odd_data)}")
print(f"statistics.median (even): {statistics.median(even_data)}")

# statistics also offers:
# - median_low()  → returns the lower of the two middle values (for even n)
# - median_high() → returns the higher of the two middle values
# - median_grouped() → for continuous/grouped data
print(f"median_low  (even): {statistics.median_low(even_data)}")
print(f"median_high (even): {statistics.median_high(even_data)}")

# Learning Point: statistics.median() is the safest choice for correctness.
# It handles edge cases like single-element lists, float precision, etc.

---
## Approach 4: Without Sorting — Quickselect Algorithm (O(n) Average)

**The Big Idea:** We don't need to fully sort the list to find the median. We just need the element that *would be* at the middle position if the list were sorted. This is the **k-th smallest element** problem.

**Quickselect** (invented by Tony Hoare, 1961) works like Quicksort but only recurses into the partition that contains the desired element.

### How It Works
1. Pick a **pivot** element.
2. **Partition** the list into elements `< pivot`, `== pivot`, and `> pivot`.
3. If the k-th element falls in the left partition, recurse left. If right, recurse right. If it's the pivot, we're done.

**Time Complexity:** O(n) average, O(n²) worst case (mitigated by random pivot).  
**Space Complexity:** O(n) for the partitions (can be O(1) with in-place version).

In [ ]:
## Quickselect — find the k-th smallest element without full sorting

import random

def quickselect(arr: list, k: int) -> int:
    """
    Find the k-th smallest element (0-indexed) in an unsorted list.
    Average Time: O(n). Does NOT sort the entire list.
    """
    if len(arr) == 1:
        return arr[0]

    # Pick a random pivot to avoid worst-case O(n²) on already-sorted data
    pivot = random.choice(arr)

    # Partition into three groups
    lows    = [x for x in arr if x < pivot]     # Elements smaller than pivot
    highs   = [x for x in arr if x > pivot]     # Elements larger than pivot
    pivots  = [x for x in arr if x == pivot]    # Elements equal to pivot

    if k < len(lows):
        # k-th smallest is in the left partition
        return quickselect(lows, k)
    elif k < len(lows) + len(pivots):
        # k-th smallest IS the pivot
        return pivot
    else:
        # k-th smallest is in the right partition
        return quickselect(highs, k - len(lows) - len(pivots))


def median_quickselect(data: list) -> float:
    """Calculate median using Quickselect — O(n) average without sorting."""
    if not data:
        raise ValueError("Cannot compute median of empty list")

    n = len(data)
    mid = n // 2

    if n % 2 == 1:
        return float(quickselect(data, mid))
    else:
        # For even length, we need both middle elements
        lower = quickselect(data, mid - 1)
        upper = quickselect(data, mid)
        return (lower + upper) / 2


# Test
odd_data = [9, 3, 7, 5, 1, 8, 2]
even_data = [9, 3, 7, 5, 1, 8]

print(f"Quickselect median (odd) : {median_quickselect(odd_data)}")
print(f"Quickselect median (even): {median_quickselect(even_data)}")

# Verify against sorted approach
print(f"Sorted verify      (odd) : {sorted(odd_data)[len(odd_data)//2]}")
print(f"Sorted verify      (even): {(sorted(even_data)[2] + sorted(even_data)[3]) / 2}")

---
## Approach 5: Using `heapq` — Two-Heap Median (Streaming Data)

What if data arrives **one element at a time** (streaming) and you need the running median? Sorting every time would be O(n² log n) total. Instead, maintain two heaps.

**Time Complexity per insertion:** O(log n)  
**Time to get median:** O(1)  
**Total for n elements:** O(n log n) — same as sorting, but works on **streaming data**.

In [ ]:
## Two-Heap median for streaming data

import heapq

class RunningMedian:
    """Maintain a running median as elements are added one at a time.
    Uses two heaps:
      - max_heap (inverted): stores the smaller half
      - min_heap: stores the larger half
    The median is always at the top of one or both heaps.
    """
    def __init__(self):
        self.max_heap = []    # Stores negated values (Python only has min-heap)
        self.min_heap = []

    def add(self, num):
        # Always add to max_heap first (negate because Python has min-heap only)
        heapq.heappush(self.max_heap, -num)

        # Ensure max_heap's largest <= min_heap's smallest
        if self.min_heap and (-self.max_heap[0] > self.min_heap[0]):
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)

        # Balance sizes: max_heap can have at most 1 more element
        if len(self.max_heap) > len(self.min_heap) + 1:
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)
        elif len(self.min_heap) > len(self.max_heap):
            val = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -val)

    def median(self) -> float:
        if len(self.max_heap) > len(self.min_heap):
            return float(-self.max_heap[0])
        else:
            return (-self.max_heap[0] + self.min_heap[0]) / 2


# Simulate streaming data
stream = [9, 3, 7, 5, 1, 8, 2]
rm = RunningMedian()

for value in stream:
    rm.add(value)
    print(f"Added {value} → Running median: {rm.median():.1f}")

# Learning Point: This is an interview favourite — "find the median of a data
# stream". It demonstrates heap mastery and is used in real-time analytics
# (monitoring server latency, stock price feeds, sensor data, etc.).

---
## Comparison of All Approaches

| Approach | Time Complexity | Space | Sorts Data? | Best For |
|---|---|---|---|---|
| Bubble Sort + index | O(n²) | O(1) in-place | Yes | Learning sorting from scratch |
| `sorted()` + index | O(n log n) | O(n) | Yes | Everyday use, small-medium data |
| `statistics.median()` | O(n log n) | O(n) | Yes | Correctness, readability |
| Quickselect | **O(n) avg** | O(n) | **No** | Large data, when sorting is wasteful |
| Two-Heap (streaming) | O(log n) per add | O(n) | No | Real-time / streaming data |

### Optimal Perspective

**For everyday code:** Use `statistics.median()` or `sorted()` + index. Clear, correct, fast enough for most datasets.

**For large datasets (millions of elements):** Quickselect avoids the full O(n log n) sort. In practice, NumPy's `np.median()` uses a C-level partial sort (introselect) that achieves O(n) — the best of both worlds.

**For streaming/real-time data:** The Two-Heap approach is the standard solution. No other approach gives O(log n) insertion with O(1) median retrieval.

**For interviews:** Know the sorted approach (shows basics), Quickselect (shows algorithm knowledge), and Two-Heap (shows data structure mastery). Mentioning the trade-offs between them demonstrates the depth interviewers look for.

> **Key Takeaway:** The progression from O(n²) → O(n log n) → O(n) mirrors a fundamental principle in computer science: *you can often do better than the obvious approach by being smarter about what work you actually need to do*. Full sorting is overkill when you only need one positional element.